In [ ]:
from langchain_core.documents import Document

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

pdf_path = "../Data/Embedding_Models.pdf"
loader = PyMuPDFLoader(pdf_path)
pages = loader.load()

print(f"Loaded {len(pages)} pages using PyMuPDF.")
pages


In [ ]:
import sys
# Add parent directory to path so python can find the 'src' folder
sys.path.append("..") 

from src.data_loader import load_multi_source_data

# 1. Define the files and databases you want to load
pdf_list = ["../Data/Embedding_Models.pdf"]
text_list = []  # Leave empty if you don't have text files yet
csv_list = []   # Leave empty if you don't have CSV files yet

db_configs = [
    # {
    #     "uri": "sqlite:///../Data/my_company_db.db", 
    #     "query": "SELECT first_name, last_name, customer_notes FROM customers;"
    # }
]

# 2. Run the helper function
all_documents = load_multi_source_data(
    pdf_files=pdf_list,
    text_files=text_list,
    csv_files=csv_list,
    db_configs=db_configs
)

print(f"Total documents parsed: {len(all_documents)}")


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# 2. Separate documents based on their source file extension
unstructured_docs = []
structured_docs = []

for doc in all_documents:
    # Get the file path or source name from metadata
    source = doc.metadata.get("source", "").lower()
    
    # If it is a PDF or Text file, we want to split it
    if source.endswith(".pdf") or source.endswith(".txt") or source.endswith(".md"):
        unstructured_docs.append(doc)
    else:
        # CSVs and DB records are kept as-is
        structured_docs.append(doc)

# 3. Split only the unstructured documents (PDFs & TXT)
split_chunks = text_splitter.split_documents(unstructured_docs)

# 4. Combine the split chunks with the untouched CSV/DB records
final_chunks = split_chunks + structured_docs

print(f"Original documents loaded: {len(all_documents)}")
print(f"Split PDF/TXT Chunks created: {len(split_chunks)}")
print(f"Untouched CSV/DB rows: {len(structured_docs)}")
print(f"Total unified chunks: {len(final_chunks)}")


In [ ]:
import uuid
from typing import List, Dict, Any, Tuple

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from sklearn.metrics.pairwise import cosine_similarity

print("✅ All RAG embedding and vector DB dependencies successfully imported!")


In [ ]:
import sys
# Tell Python to search the parent directory for the 'src' package
sys.path.append("..") 

from src.embedding_manager import EmbeddingManager

# Initialize Google Gemini embedding model
embedder = EmbeddingManager(provider="google")

# Test generating a vector
sample_vector = embedder.generate_embedding("What is the company leave policy?")

print(f"✅ Generated Vector Length (Dimensions): {len(sample_vector)}")


In [1]:
import sys
sys.path.append("..")

from src.data_loader import load_multi_source_data
from src.embedding_manager import EmbeddingManager
from src.vector_store import VectorStoreManager
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load raw data
raw_docs = load_multi_source_data(pdf_files=["../Data/Embedding_Models.pdf"])

# 2. Chunk documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
final_chunks = text_splitter.split_documents(raw_docs)

# 3. Initialize Google Gemini (1024 dims) & Pinecone DB
embedder = EmbeddingManager(provider="google", dimension=1024)
vector_db = VectorStoreManager(provider="pinecone")

# 4. Generate 1024-dim embeddings & upsert to Cloud Pinecone!
chunk_texts = [doc.page_content for doc in final_chunks]
chunk_vectors = embedder.generate_embeddings(chunk_texts)

vector_db.add_documents(documents=final_chunks, embeddings=chunk_vectors)


/opt/homebrew/Caskroom/miniforge/base/envs/rag_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading PDF: ../Data/Embedding_Models.pdf...
--- Ingestion Complete: Unified 10 document pages/rows ---
Loading Google Gemini embedding model: 'models/gemini-embedding-001'...
✅ Google Gemini Embedding Model initialized successfully! (1024 dimensions)
✅ Connected to Cloud Pinecone Index: 'elegant-walnut'
✅ Upserted 69 vectors to Cloud Pinecone index.
